In [ ]:
import json
import requests
import polars as pl
import os
from dotenv import load_dotenv
import boto3
load_dotenv()

# --- CONFIGURATION ---
INPUT_S3_PATH = ""
# print(os.getenv("S3_BUCKET_NAME"))
OPENALEX_API_KEY = ""  # Optional: paste your free key from openalex.org/settings/api
POLITE_EMAIL = "esdionneca@gmail.com"

KEEP_COLS = ["document_id", "uuid", "pdf_name", "s3_key"]

In [ ]:
import polars as pl

storage_options = {

}

df = pl.read_parquet(
    f"s3://{INPUT_S3_PATH}/metadata_table/metadata_20261110-223144.parquet",
    storage_options=storage_options,
)

In [ ]:
# Pull the 4 kept columns + 'title' for 1 valid row
sample_df = (
    pl.scan_parquet(f"s3://{INPUT_S3_PATH}/metadata_table/metadata_20261110-223144.parquet", storage_options=storage_options)
    .select(KEEP_COLS + ["title"])
    .filter(pl.col("title").is_not_null() & (pl.col("title").str.strip_chars() != ""))
    .limit(1)
    .collect()
)

row = sample_df.to_dicts()[0]
print("Source S3 Metadata Row:")
print(json.dumps(row, indent=2, default=str))

In [ ]:
title_query = row["title"].strip()

# OpenAlex uses commas to separate multiple filters, so strip commas inside the title value
clean_title = title_query.replace(",", " ")

params = {
    "filter": f"title.search:{clean_title}",
    "per_page": 1,
    "mailto": POLITE_EMAIL,
}
if OPENALEX_API_KEY:
    params["api_key"] = OPENALEX_API_KEY

resp = requests.get("https://api.openalex.org/works", params=params, timeout=20)
resp.raise_for_status()

data = resp.json()
results = data.get("results", [])

if not results:
    print(f"No OpenAlex results found for: {title_query!r}")
    oa_work = {}
else:
    oa_work = results[0]
    print(f"Matched OpenAlex Title : {oa_work.get('title')}")
    print(f"OpenAlex ID            : {oa_work.get('id')}")
    print(f"DOI                    : {oa_work.get('doi')}")
    print(f"Relevance Score        : {oa_work.get('relevance_score')}")

In [ ]:
print(f"{'OPENALEX KEY':<30} | {'TYPE':<10} | {'VALUE PREVIEW'}")
print("-" * 95)

for key, val in oa_work.items():
    val_type = type(val).__name__
    preview = json.dumps(val, ensure_ascii=False)
    if len(preview) > 55:
        preview = preview[:52] + "..."
    print(f"{key:<30} | {val_type:<10} | {preview}")

In [ ]:
ids_dict = oa_work.get("ids") or {}
primary_source = (oa_work.get("primary_location") or {}).get("source") or {}

selected_oa_fields = {
    "openalex_id": oa_work.get("id"),
    "doi": oa_work.get("doi"),
    "mag_id": ids_dict.get("mag"),
    "pmid": ids_dict.get("pmid"),
    "openalex_title": oa_work.get("title"),
    "publication_year": oa_work.get("publication_year"),
    "publication_date": oa_work.get("publication_date"),
    "type": oa_work.get("type"),
    "language": oa_work.get("language"),
    "cited_by_count": oa_work.get("cited_by_count"),
    "is_oa": (oa_work.get("open_access") or {}).get("is_oa"),
    "oa_status": (oa_work.get("open_access") or {}).get("oa_status"),
    "oa_url": (oa_work.get("open_access") or {}).get("oa_url"),
    "source_name": primary_source.get("display_name"),
    "issn_l": primary_source.get("issn_l"),
    "authors": [
        a.get("author", {}).get("display_name")
        for a in (oa_work.get("authorships") or [])
    ],
    "primary_topic": (oa_work.get("primary_topic") or {}).get("display_name"),
    "relevance_score": oa_work.get("relevance_score"),
}

merged_record = {k: row[k] for k in KEEP_COLS} | selected_oa_fields

enriched_df = pl.DataFrame([merged_record])
display(enriched_df)